# SO(3) and O(3) — Companion Notebook

**6.7970/8.750 Symmetry and its Application to Machine Learning**

This notebook follows the SO(3) and O(3) exercise section by section.
Use it to **prototype your code** and **test your implementations**
against the course library before submitting on the website.

Each section includes small tests you can use to check your work.
In between the problem cells, you'll also find **context cells** with
demonstrations and visualizations designed to build intuition — these
are not graded, but are worth reading and running before diving into
the problems.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atomicarchitects/symm4ml-colabs/blob/main/so3_companion.ipynb)

## Setup

In [ ]:
%%capture
!pip install https://symm4ml.mit.edu/_static/symm4ml_s26/symm4ml/symm4ml_latest.zip

In [ ]:
from typing import List

import numpy as np
import scipy.linalg

from symm4ml import groups, linalg, rep, lie, so3, grids, vis, vib_modes

### Reference data

These arrays are used throughout the exercise for testing.

In [ ]:
# SO(3) L=1 generators
so3_gens = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)

# SO(3) irreps up to L=4 (used by several tests and explorations)
so3_irreps = lie.infer_irreps_from_tensor_products(so3_gens, 5)

# Rotation and inversion parameters for O(3) irrep testing
rot_params = np.array(
    [[ -3.77392419,  -1.83276588,  -3.78063415],
     [ 11.63820707,  -0.08480556,  -6.64579377],
     [  5.16820211,  -7.67078688,   1.31232867],
     [-12.31297053,  -8.34523907,   1.23691562],
     [  4.63992237,   1.07673867,  -0.72663959],
     [ -1.89189032,  -9.28982765,  -4.52291455],
     [ -2.89427876,   6.64209484,   2.15901739],
     [-11.077508  ,   2.03627963,  -2.41954333],
     [ -4.25322637,   3.84327547,   6.47796105],
     [  5.85140556,  -5.27295921,  -1.94283866]]
)
inv_params = np.array([1.] * 5 + [-1] * 5)

print(f"so3_irreps: {len(so3_irreps)} irreps, dimensions: {[X.shape[-1] for X in so3_irreps]}")

### Building SO(3) irreps from generators

The function `lie.infer_irreps_from_tensor_products(generators, n)` is the key workhorse for constructing SO(3) irreps. Starting from the $L=1$ generators, it builds higher-$L$ irreps by taking successive tensor products and decomposing the result. The output is a list of irrep generators from $L=0$ up to $L=n-1$.

In [ ]:
# Demonstrate lie.infer_irreps_from_tensor_products
demo_irreps = lie.infer_irreps_from_tensor_products(so3_gens, 5)
for L, ir in enumerate(demo_irreps):
    print(f"L={L}: generators shape {ir.shape}, dimension {ir.shape[-1]}")

---
## Section 1: Representations of SO(3) and O(3)

### 1. `axis_angle_to_matrix(axis, angle)`

Convert an axis-angle representation to a rotation matrix using the matrix exponential.

**Hint:** A rotation around an arbitrary axis can be written $e^{\theta L_{\text{axis}}}$. Find $L_{\text{axis}}$ in terms of $L_x$, $L_y$ and $L_z$ for the $L=1$ irrep (stored in `so3_gens`).

In [ ]:
def axis_angle_to_matrix(axis: np.ndarray, angle: float) -> np.ndarray:
    """Convert an axis-angle representation to a rotation matrix.
    Input:
        axis: a unit vector representing the axis of rotation, np.array of shape [3]
        angle: the angle of rotation, in radians
    Output:
        R: a 3x3 rotation matrix, np.array [3, 3]
    """
    assert axis.shape == (3,)
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from so3.py
np.testing.assert_allclose(
    axis_angle_to_matrix(np.array([0.0, 0.0, 1.0]), np.pi / 2.0),
    np.array([[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 1.0]]),
)
print("axis_angle_to_matrix tests passed!")

### 2. `o3_irrep(so3_irrep_gen, rot_parameters, inv_parameters, parity)`

Compute O(3) irrep matrices from SO(3) generators, rotation parameters, and inversion parameters. $O(3)$ is the direct product of $SO(3)$ and inversion ($Z_2$), so its irreps are products of the irreps of these individual groups.

In [ ]:
def o3_irrep(so3_irrep_gen, rot_parameters, inv_parameters, parity):
    """Compute the O(3) irrep matrices for given SO(3) irrep
    generators, rotation, and inversion parameters.

    Inputs:
        so3_irrep_gen: (ndarray) SO(3) irreducible representation
            generators. Shape (n, i, j).
        rot_parameters: (ndarray) Rotation parameters, shape (m, n).
        inv_parameters: (ndarray) Inversion parameters, shape (m,).
        parity: (int) 1 for even, -1 for odd parity.

    Output:
        ndarray: Shape (m, i, j) O(3) representation matrices.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in so3.py for o3_irrep.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    o3_irrep(so3_irreps[1], rot_params, inv_params, parity=1),
    so3.o3_irrep(so3_irreps[1], rot_params, inv_params, parity=1),
    atol=1e-7
)
np.testing.assert_allclose(
    o3_irrep(so3_irreps[1], rot_params, inv_params, parity=-1),
    so3.o3_irrep(so3_irreps[1], rot_params, inv_params, parity=-1),
    atol=1e-7
)
print("o3_irrep comparison passed!")

### Commonly known irreps of $O(3)$

$O(3)$, the group of 3D rotoinversions, is a direct product group of 3D rotations and inversion, i.e. $O(3) = SO(3) \times Z_2$, which commute with each other.
Because of this, we know the irreps of this group are simple combinations of the irreps of the individual groups (Dresselhaus Ch 6.3). Thus, irreps are distinguished by two indices, $L$ from $SO(3)$ and parity $p$ which is either even (1) or odd (-1). We will also use the shorthand `Lp` to denote irreps of $O(3)$ which matches the convention in the [python library `e3nn`](https://e3nn.org/).

* `0e`: A scalar, e.g. classification labels
* `0o`: A pseudoscalar, e.g. a "right" hand
* `1o`: A vector - angular frequency $L=1$ and flips under inversion e.g. 3D coordinates or forces
* `1e`: A pseudovector - angular frequency $L=1$ and does not flip under inversion, e.g. a rotation axis or a magnetic field

While the $SO(3)$ part of $O(3)$ forms a Lie group, $Z_2$ does not.
In order to determine how potentially reducible representations decompose into $O(3)$ irreps, we will use the representations (rather than generators) of this group. In cases where we need to "span" the entire group of $O(3)$, we can simply generate the matrix representations of $O(3)$ over random 3D rotoinversions.

### 3. `clebsch_gordan(in1_irrep_gen, in2_irrep_gen, out_irrep_gen)`

Compute and normalize the Clebsch-Gordan coefficients for a triple of irrep generators.

In [ ]:
def clebsch_gordan(in1_irrep_gen, in2_irrep_gen, out_irrep_gen, tol=1e-8):
    """Compute and normalize the Clebsch-Gordan coefficients for specified
    irrep generators of same group.

    Inputs:
        in1_irrep_gen: (ndarray) First input irrep generators
        in2_irrep_gen: (ndarray) Second input irrep generators
        out_irrep_gen: (ndarray) Output irrep generators

    Output:
        (ndarray) Shape (d1, d2, d3) normalized Clebsch-Gordan coefficients.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in so3.py for clebsch_gordan.
# Supplementary comparison with course (not a course small test):
# Note: CG coefficients have a sign ambiguity, so we compare absolute values.
np.testing.assert_allclose(
    np.abs(clebsch_gordan(so3_irreps[1], so3_irreps[1], so3_irreps[0])),
    np.abs(so3.clebsch_gordan(so3_irreps[1], so3_irreps[1], so3_irreps[0])),
    atol=1e-7
)
np.testing.assert_allclose(
    np.abs(clebsch_gordan(so3_irreps[1], so3_irreps[1], so3_irreps[2])),
    np.abs(so3.clebsch_gordan(so3_irreps[1], so3_irreps[1], so3_irreps[2])),
    atol=1e-7
)
print("clebsch_gordan comparison passed!")

### Properties of Clebsch-Gordan Coefficients

#### Selection rules
[Clebsch-Gordan Coefficients](https://en.wikipedia.org/wiki/Clebsch%E2%80%93Gordan_coefficients) are only non-zero if the following inequality is satisfied:

$|L_{\text{in1}} - L_{\text{in2}}| \leq L_{\text{out}}\leq L_{\text{in1}} + L_{\text{in2}}$

This equation articulates the "selection rules" of $SO(3)$.

**We shall call each nontrivial combination of $(L_{\text{in1}}, L_{\text{in2}}, L_{\text{out}})$ a "path"**.

For example, $(1, 2, 3)$ satisfies this equation but $(1, 2, 4)$ does not.

In [ ]:
L0, L1, L2, L3, L4 = so3_irreps[0], so3_irreps[1], so3_irreps[2], so3_irreps[3], so3_irreps[4]
cg_L1_L2_L3 = so3.clebsch_gordan(L1, L2, L3)
cg_L3_L2_L1 = so3.clebsch_gordan(L3, L2, L1)
cg_L2_L1_L3 = so3.clebsch_gordan(L2, L1, L3)
cg_L2_L1_L4 = so3.clebsch_gordan(L2, L1, L4)

print("CG(L2, L1, L3) is trivial?", np.allclose(np.zeros_like(cg_L2_L1_L3), cg_L2_L1_L3))  # False: |2-1|<=3<=2+1
print("CG(L2, L1, L4) is trivial?", np.allclose(np.zeros_like(cg_L2_L1_L4), cg_L2_L1_L4))  # True: 4 > 2+1

#### Clebsch-Gordan coefficients are equivariant under permutation of the two inputs, but not an input and output.

In [ ]:
# We can swap the inputs and arrive at the same change of basis
print("CG(L1,L2,L3) == CG(L2,L1,L3).transpose?", np.allclose(cg_L1_L2_L3, cg_L2_L1_L3.transpose(1, 0, 2)))

# But this is not the case if we permute an output with an input
print("CG(L1,L2,L3) == CG(L3,L2,L1).transpose?", np.allclose(cg_L1_L2_L3, cg_L3_L2_L1.transpose(2, 1, 0)))

#### Clebsch-Gordan Coefficients are normalized such that the output over all possible **paths** will have unit norm for two inputs of unit norm.

For example, $1 \otimes 1 = 0 \oplus 1 \oplus 2$. Given two normalized inputs, we will recover a norm 1 output over all paths.

In [ ]:
cg_L1_L1_L0 = so3.clebsch_gordan(L1, L1, L0)
cg_L1_L1_L1 = so3.clebsch_gordan(L1, L1, L1)
cg_L1_L1_L2 = so3.clebsch_gordan(L1, L1, L2)
cg_L1_L1 = np.concatenate([cg_L1_L1_L0, cg_L1_L1_L1, cg_L1_L1_L2], axis=-1)

all_paths = np.einsum('ijk,i,j->k', cg_L1_L1, np.array([1, 0, 0]), np.array([0, 1, 0]))
print("norm([1,0,0] x [0,1,0] over all paths):", np.linalg.norm(all_paths))

all_paths = np.einsum('ijk,i,j->k', cg_L1_L1, np.array([1, 0, 1]) * 1 / np.sqrt(2), np.array([1, 0, -1]) * 1 / np.sqrt(2))
print("norm(mixed inputs over all paths):", np.linalg.norm(all_paths))

Note: [Wigner 3j symbols](https://en.wikipedia.org/wiki/3-j_symbol) are similar to Clebsch-Gordan coefficients, they just differ by normalization scheme. Wigner 3j symbols do not normalize for the output to preserve unit norm, but instead have the property that if we transpose the tensor product indices we arrive at the same values.

### 4. `spherical_harmonics(Xs, x)`

Compute spherical harmonics by successive tensor products and reductions into irreps:

$$Y^l(\vec x) = Q_l (\underbrace{\vec x \otimes \vec x \otimes \dots \otimes \vec x}_{l \text{ times}})$$

Normalize each output to have unit norm.

In [ ]:
def spherical_harmonics(Xs: List[np.ndarray], x: np.ndarray) -> List[np.ndarray]:
    """Compute the spherical harmonics of a vector.
    Input:
        Xs: list of irreps of SO(3), from l=0 to l=L
            (each an np.array [3, 2*l+1, 2*l+1])
        x: a vector in R^3, np.array of size [3]
    Output:
        sh: list of spherical harmonics evaluated at x, from l=0 to l=L
            (each an np.array [2*l+1])
    Note:
        Normalize each output to have unit norm.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Small tests from so3.py
X = np.array(
    [
        [[0.0, 0.0, 0.0], [0.0, 0.0, -1.0], [0.0, 1.0, 0.0]],
        [[0.0, 0.0, 1.0], [0.0, 0.0, 0.0], [-1.0, 0.0, 0.0]],
        [[0.0, -1.0, 0.0], [1.0, 0.0, 0.0], [0.0, 0.0, 0.0]],
    ]
)
Xs_len3 = lie.infer_irreps_from_tensor_products(X, 3)
x = np.array([0.0, 1.0, 0.0])
np.testing.assert_allclose(spherical_harmonics(Xs_len3, x)[0], 1.0)
np.testing.assert_allclose(spherical_harmonics(Xs_len3, x)[1], x)
print("spherical_harmonics tests passed!")

### Spherical Harmonics from tensor products

Compare to [Real Spherical Harmonics](https://en.wikipedia.org/wiki/Table_of_spherical_harmonics#Real_spherical_harmonics).

We can visualize what the CG coefficients look like as polynomials. Because of the properties of similarity transforms ($U e^{X}U^{-1} = e^{UXU^{-1}}$), we can compute the change of basis between tensor product representations and reshape to get the Clebsch-Gordan coefficients.

In [ ]:
# Build O(3) representation matrices from random rotoinversions
np.random.seed(42)
N = 10
rand_rots = np.random.rand(N, 3) * 2 * np.pi
rand_signs = np.array([1] * int(N/2) + [-1] * int(N/2))

o3_irrep_mats = []
o3_irrep_labels = []
Lmax = 4
for L in range(Lmax+1):
    for p in [1, -1]:
        o3_irrep_mats.append(so3.o3_irrep(so3_irreps[L], rand_rots, rand_signs, p))
        o3_irrep_labels.append("{L}{p}".format(L=L, p="e" if p==1 else "o"))

print(o3_irrep_labels)

# Because we are dealing with representations instead of generators,
# we go back to using functions from `rep` instead of `lie`.
o3_1o = o3_irrep_mats[o3_irrep_labels.index('1o')]
o3_1o_1o = rep.tensor_product(o3_1o, o3_1o)

#### $L=2$ spherical harmonics as polynomials

$(L=1) \otimes (L=1) = (L=0) \oplus (L=1) \oplus (L=2)$

We can extract the $L=2$ piece via `infer_change_of_basis`, reshape the change-of-basis matrix to $(3, 3, 5)$, and contract with symbolic variables to see the resulting polynomials.

In [ ]:
import sympy as sp

cob = linalg.infer_change_of_basis(o3_1o_1o, o3_irrep_mats[4])  # 4 = index of '2e'
tensor = np.einsum('ijk->kij', cob.reshape(3, 3, 5)).round(3)

x, y, z = sp.symbols('x y z')
variables = [x, y, z]

print("The 5 polynomials that transform as L=2:")
for t in tensor:
    poly_expr = sum(t[i, j] * variables[i] * variables[j]
                    for i in range(3) for j in range(3))
    print(" ", sp.expand(poly_expr))

#### Visualizing the CG coefficient matrices

We can also visualize the change-of-basis matrices as images. Each matrix below shows how one component of the $L=2$ output is built from the $3 \times 3$ tensor product of two $L=1$ inputs:

In [ ]:
cob_2e = linalg.infer_change_of_basis(o3_1o_1o, o3_irrep_mats[o3_irrep_labels.index('2e')])
vis.plot_matrices(np.einsum('ijk->kij', cob_2e.reshape(3, 3, 5)), cmap='RdBu');

The 5 polynomials that transform as $L=2$ (neglecting normalization) are:

$\{2x^2 - y^2 - z^2, \; xy, \; zx, \; y^2 - z^2, \; yz\}$

These are a valid choice of $L=2$ spherical harmonics. (You can see that they are similar to those [here](https://en.wikipedia.org/wiki/Table_of_spherical_harmonics#Real_spherical_harmonics) up to a change of basis.)

---
## Section 2: Functions for Decomposing Cartesian Tensors

### 5. `formula_to_perm_and_sign_group(formula)`

Generate a permutation group from an einsum-style index formula. The left-hand side indicates the original index order and the right-hand side the permuted order. A preceding `-` on an index indicates a sign change.

In [ ]:
def formula_to_perm_and_sign_group(formula):
    """Generate a permutation group from an einsum-style index formula.

    Input:
        formula: (str) e.g. 'ij=-ji'

    Output:
        Tuple of (perm_rep, sign_rep):
        - perm_rep: np.array (G, d, d) permutation matrices
        - sign_rep: np.array (G,) sign changes on values
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in grids.py for formula_to_perm_and_sign_group.
# Supplementary comparison with course (not a course small test):
perm, sign = formula_to_perm_and_sign_group('ij=-ji')
perm_ref, sign_ref = grids.formula_to_perm_and_sign_group('ij=-ji')
np.testing.assert_allclose(perm, perm_ref, atol=1e-7)
np.testing.assert_allclose(sign, sign_ref, atol=1e-7)

perm2, sign2 = formula_to_perm_and_sign_group('ijk=-jik=-ikj')
perm2_ref, sign2_ref = grids.formula_to_perm_and_sign_group('ijk=-jik=-ikj')
np.testing.assert_allclose(perm2, perm2_ref, atol=1e-7)
np.testing.assert_allclose(sign2, sign2_ref, atol=1e-7)
print("formula_to_perm_and_sign_group comparison passed!")

### 6. `perm_to_grid_rep(perm_rep, dims)`

Construct the representation of a permutation group on grid indices.

In [ ]:
def perm_to_grid_rep(perm_rep, dims, eps=1e-6):
    """Construct the representation of a permutation group on grid indices.

    Input:
        perm_rep: (np.array) Shape (G, d, d) permutation matrices.
        dims: (iterable of int) Dimensions of the tensor.

    Output:
        np.array of shape (G, dof, dof) permutation representation on grid.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in grids.py for perm_to_grid_rep.
# Supplementary comparison with course (not a course small test):
np.testing.assert_allclose(
    perm_to_grid_rep(groups.permutation_matrices(2), [3, 3]),
    grids.perm_to_grid_rep(groups.permutation_matrices(2), [3, 3]),
    atol=1e-7
)
np.testing.assert_allclose(
    perm_to_grid_rep(groups.permutation_matrices(3), [2, 2, 2]),
    grids.perm_to_grid_rep(groups.permutation_matrices(3), [2, 2, 2]),
    atol=1e-7
)
print("perm_to_grid_rep comparison passed!")

### 7. `perm_and_sign_to_tensor_basis_and_proj(perm_rep, sign_rep, dims)`

Obtain the tensor basis and projector from the permutation group representation. Sum over group elements and apply Gram-Schmidt orthonormalization.

In [ ]:
def perm_and_sign_to_tensor_basis_and_proj(perm_rep, sign_rep, dims, tol=1e-8):
    """Obtain the tensor basis and projector from the permutation group representation.

    Input:
        perm_rep: (ndarray) Shape (G, d, d) permutation matrices.
        sign_rep: (ndarray) Shape (G,) sign changes on values.
        dims: (iterable of int) Dimensions of the tensor.
        tol: (float) Tolerance for Gram-Schmidt.

    Output:
        Two ndarrays: orthonormal tensor basis and projector.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# No small tests in grids.py for perm_and_sign_to_tensor_basis_and_proj.
# Supplementary comparison with course (not a course small test):
_perm, _sign = grids.formula_to_perm_and_sign_group('ij=ji')
_, proj = perm_and_sign_to_tensor_basis_and_proj(_perm, _sign, [3, 3])
_, proj_ref = grids.perm_and_sign_to_tensor_basis_and_proj(_perm, _sign, [3, 3])
np.testing.assert_allclose(proj, proj_ref, atol=1e-7)

_perm2, _sign2 = grids.formula_to_perm_and_sign_group('ij=-ji')
_, proj2 = perm_and_sign_to_tensor_basis_and_proj(_perm2, _sign2, [3, 3])
_, proj2_ref = grids.perm_and_sign_to_tensor_basis_and_proj(_perm2, _sign2, [3, 3])
np.testing.assert_allclose(proj2, proj2_ref, atol=1e-7)
print("perm_and_sign_to_tensor_basis_and_proj comparison passed!")

---
## Section 3: Practice Decomposing Cartesian Tensors into O(3) Irreps

Give the decompositions of the following Cartesian Tensors into irreps of $O(3)$.

Give your answers as a **list of integers** in the order `['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']`.

### Visualizing index permutations

To build intuition for `perm_to_grid_rep`, here's what the `'ij=ji'` permutation looks like on a $3 \times 3$ grid — it swaps rows and columns (i.e., transposes the matrix):

In [ ]:
perm_rep, sign_rep = grids.formula_to_perm_and_sign_group('ij=ji')
grid_rep = grids.perm_to_grid_rep(perm_rep, [3, 3])  # [3, 3] because order 2 cart tensor

vis.plot_image_values(
    (grid_rep @ np.arange(3 * 3)).reshape(-1, 3, 3),
    (grid_rep @ np.arange(3 * 3)).reshape(-1, 3, 3),
    colormap="plasma", fontcolor=['white', 'black'], fontcolor_cutoff=4);

### Workflow for tensor decomposition

The full pipeline for decomposing a Cartesian tensor with index symmetries into $O(3)$ irreps is:

1. Build the **tensor product representation** (e.g. `1o` $\otimes$ `1o` for a rank-2 tensor)
2. If the tensor has index symmetries (e.g. `'ij=ji'`), use `formula_to_perm_and_sign_group` $\to$ `perm_and_sign_to_tensor_basis_and_proj` to get the symmetry-constrained subspace basis
3. **Project** the tensor product representation into that subspace
4. **Decompose** into $O(3)$ irreps via `linalg.infer_change_of_basis`

**Parity tip:** A tensor product of $n$ vectors (`1o`, parity $= -1$) has overall parity $(-1)^n$. So rank-2 tensors (`ij`) contain only **even** irreps, rank-3 (`ijk`) only **odd**, rank-4 (`ijkl`) only **even**, etc. This lets you sanity-check your answers.

In [ ]:
# Use this cell to explore tensor decomposition.
# Workflow:
# 1. Build O(3) irreps using so3.o3_irrep with rot_params/inv_params
# 2. Build tensor representation (e.g., tensor product of 1o x 1o for 'ij')
# 3. Apply symmetry constraints using grids.formula_to_perm_and_sign_group
# 4. Decompose the result into O(3) irreps
#
# YOUR EXPLORATION HERE

In [ ]:
# YOUR ANSWER HERE:
# Decompose a 1o x 1o matrix, i.e. 'ij'.
# ij = [...]  # List of multiplicities for ['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']

In [ ]:
# YOUR ANSWER HERE:
# Decompose a symmetric 1o x 1o matrix, i.e. 'ij=ji'.
# ij_ji = [...]  # List of multiplicities for ['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']

In [ ]:
# YOUR ANSWER HERE:
# Decompose an antisymmetric 1o x 1o matrix, i.e. 'ij=-ji'.
# ij_mji = [...]  # List of multiplicities for ['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']

In [ ]:
# YOUR ANSWER HERE:
# Decompose a 1o x 1o x 1o tensor, i.e. 'ijk'.
# ijk = [...]  # List of multiplicities for ['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']

In [ ]:
# YOUR ANSWER HERE:
# Decompose a fully antisymmetric 1o x 1o x 1o tensor, i.e. 'ijk=-jik=-ikj' (Levi-Civita).
# ijk_mjik_mikj = [...]  # List of multiplicities for ['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']

In [ ]:
# YOUR ANSWER HERE:
# Decompose a 1o x 1o x 1o x 1o tensor, i.e. 'ijkl'.
# ijkl = [...]  # List of multiplicities for ['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']

In [ ]:
# YOUR ANSWER HERE:
# Decompose a 1o^4 tensor under 'ijkl=jikl=ijlk=klij' symmetries (elasticity tensor).
# ijkl_jikl_ijlk_klij = [...]  # List of multiplicities for ['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']

---
## Section 4: Branching Rules of O(3) Irreps under Point Groups

Derive the decomposition of O(3) irreps under specific point group subgroups.

Reference character tables:
- $T_d$: irreps `['A1', 'A2', 'E', 'T1', 'T2']`
- $C_{4v}$: irreps `['A1', 'A2', 'B1', 'B2', 'E']`
- $C_{2v}$: irreps `['A1', 'A2', 'B1', 'B2']`

In [ ]:
# Use this cell to explore branching rules.
# Workflow:
# 1. Build O(3) irrep representation restricted to a point group's elements
# 2. Decompose into the point group's irreps
# 3. Count multiplicities
#
# IMPORTANT: infer_irreps returns irreps in an arbitrary order. You need to
# reorder them to match the standard character table conventions used in the
# homework. The cells below set up the correct ordering for Td, C4v, C2v.
#
# YOUR EXPLORATION HERE

### Matching irrep ordering to standard character tables

The homework uses standard character table conventions for irrep ordering. The cells below reorder the output of `infer_irreps` to match:
- [$T_d$ Character Table](https://symmetry.constructor.university/cgi-bin/group.cgi?group=902&option=4): `['A1', 'A2', 'E', 'T1', 'T2']`
- [$C_{4v}$ Character Table](https://symmetry.constructor.university/cgi-bin/group.cgi?group=404&option=4): `['A1', 'A2', 'B1', 'B2', 'E']`
- [$C_{2v}$ Character Table](https://symmetry.constructor.university/cgi-bin/group.cgi?group=402&option=4): `['A1', 'A2', 'B1', 'B2']`

In [ ]:
from pymatgen.symmetry.groups import PointGroup

# --- Td ---
Td_group = PointGroup(vib_modes.point_group_dict['Td'])
Td = np.stack([op.rotation_matrix for op in Td_group.symmetry_ops], axis=0)
Td_table = groups.make_multiplication_table(Td)
np.random.seed(42)
Td_irreps = rep.infer_irreps(Td_table)
Td_conj = groups.conjugacy_classes(Td_table)

Td_conj_reordered = [list(Td_conj)[i] for i in [1, 2, 3, 4, 0]]
Td_irreps_reordered = [Td_irreps[i] for i in [0, 2, 1, 4, 3]]
Td_irrep_labels = ['A1', 'A2', 'E', 'T1', 'T2']

print("Td character table:")
print(rep.character_table(Td_irreps_reordered, Td_conj_reordered).round(2).real)

In [ ]:
# --- C4v ---
C4v_group = PointGroup(vib_modes.point_group_dict['C4v'])
C4v = np.stack([op.rotation_matrix for op in C4v_group.symmetry_ops], axis=0)
C4v_table = groups.make_multiplication_table(C4v)
np.random.seed(42)
C4v_irreps = rep.infer_irreps(C4v_table)
C4v_conj = groups.conjugacy_classes(C4v_table)

C4v_conj_reordered = [list(C4v_conj)[i] for i in [-2, 0, 2, 4, 1]]
C4v_irreps_reordered = [C4v_irreps[i] for i in [0, 1, 3, 2, 4]]
C4v_irrep_labels = ['A1', 'A2', 'B1', 'B2', 'E']

print("C4v character table:")
print(rep.character_table(C4v_irreps_reordered, C4v_conj_reordered).round(2).real)

In [ ]:
# --- C2v ---
C2v_group = PointGroup(vib_modes.point_group_dict['C2v'])
C2v = np.stack([op.rotation_matrix for op in C2v_group.symmetry_ops], axis=0)
C2v_table = groups.make_multiplication_table(C2v)
np.random.seed(42)
C2v_irreps = rep.infer_irreps(C2v_table)
C2v_conj = groups.conjugacy_classes(C2v_table)

C2v_conj_reordered = [list(C2v_conj)[i] for i in [1, 0, 2, 3]]
C2v_irreps_reordered = [C2v_irreps[i] for i in [3, 2, 0, 1]]
C2v_irrep_labels = ['A1', 'A2', 'B1', 'B2']

print("C2v character table:")
print(rep.character_table(C2v_irreps_reordered, C2v_conj_reordered).round(2).real)

### Helper: `o3_branch` — explore branching rules interactively

The helper below builds O(3) irreps restricted to a point group and decomposes them via `vib_modes.branching_change_of_basis`. Use it to quickly explore how O(3) irreps branch under any subgroup:

In [ ]:
def o3_branch(so3_irreps, sub, sub_irreps, sub_irrep_labels):
    """Branch O(3) irreps (up to L=3, both parities) into a point group's irreps.

    Args:
        so3_irreps: list of SO(3) irrep generators (from lie.infer_irreps_from_tensor_products)
        sub: (|H|, 3, 3) array of point group elements (3x3 matrices)
        sub_irreps: list of H-irrep matrices
        sub_irrep_labels: list of label strings for H-irreps
    Returns:
        branch: nested list of change-of-basis matrices
    """
    rot_params, sign_params = so3.vec_3d_rep_to_rot_and_sign_params(sub)
    o3_irreps = [
        so3.o3_irrep(so3_irreps[i], rot_params, sign_params, parity=p)
        for i in range(len(so3_irreps)) for p in [1, -1]
    ]
    o3_irrep_labels = ["{L}{p}".format(L=L, p=p)
                       for L in range(len(so3_irreps)) for p in ["e", "o"]]
    branch = vib_modes.branching_change_of_basis(
        o3_irreps, sub_irreps, np.arange(len(sub))
    )
    print('\t' + '\t\t'.join(sub_irrep_labels))
    for i, label in enumerate(o3_irrep_labels):
        print(label, "\t", "\t".join(
            [str(branch[i][j].shape) for j in range(len(sub_irrep_labels))]
        ))
    return branch

# Example: branch O(3) irreps under Td
print("O(3) branching under Td:")
_ = o3_branch(so3_irreps, Td, Td_irreps_reordered, Td_irrep_labels)

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 1o decompose under Td?
# o3_1o_under_Td = [...]  # Multiplicities in order ['A1', 'A2', 'E', 'T1', 'T2']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 1e decompose under Td?
# o3_1e_under_Td = [...]  # Multiplicities in order ['A1', 'A2', 'E', 'T1', 'T2']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 2e decompose under Td?
# o3_2e_under_Td = [...]  # Multiplicities in order ['A1', 'A2', 'E', 'T1', 'T2']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 1o decompose under C4v?
# o3_1o_under_C4v = [...]  # Multiplicities in order ['A1', 'A2', 'B1', 'B2', 'E']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 1e decompose under C4v?
# o3_1e_under_C4v = [...]  # Multiplicities in order ['A1', 'A2', 'B1', 'B2', 'E']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 2e decompose under C4v?
# o3_2e_under_C4v = [...]  # Multiplicities in order ['A1', 'A2', 'B1', 'B2', 'E']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 1o decompose under C2v?
# o3_1o_under_C2v = [...]  # Multiplicities in order ['A1', 'A2', 'B1', 'B2']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 1e decompose under C2v?
# o3_1e_under_C2v = [...]  # Multiplicities in order ['A1', 'A2', 'B1', 'B2']

In [ ]:
# YOUR ANSWER HERE:
# How does the O(3) irrep 2e decompose under C2v?
# o3_2e_under_C2v = [...]  # Multiplicities in order ['A1', 'A2', 'B1', 'B2']

---
## Section 5: Branching Change of Basis

Implement the general machinery for branching: given irreps of a group $G$ and irreps of a subgroup $H \leq G$, decompose each $G$-irrep into a direct sum of $H$-irreps.

This is just `linalg.infer_change_of_basis` applied per (G-irrep, H-irrep) pair, where $S$ is the $G$-irrep restricted to the elements that form $H$ and $R$ is the $H$-irrep.

**Note**: This function lives in `symm4ml/vib_modes.py` (not `so3.py`) because its tests use point groups from `pymatgen`.

In [ ]:
def branching_change_of_basis(G_irreps, H_irreps, H_indices):
    """Compute the change-of-basis matrices for branching G-irreps into H-irreps.

    Inputs:
        G_irreps: list of G-irrep matrices, each of shape (|G|, d_G, d_G)
        H_irreps: list of H-irrep matrices, each of shape (|H|, d_H, d_H)
        H_indices: list/array of indices into G's elements that form H

    Output:
        List of lists of change-of-basis matrices Q,
        where Q[i][j] has shape (multiplicity, d_H, d_G) for the
        j-th H-irrep inside the i-th G-irrep.
    """
    # YOUR CODE HERE
    pass

In [ ]:
# Test branching_change_of_basis using Td -> C2v branching
# The subgroup elements of C2v inside Td are at indices [10, 6, 1, 23]
_subs_elements = [10, 6, 1, 23]

np.random.seed(42)
Td_irreps = rep.infer_irreps(vib_modes.Td_table)
C2v_irreps = rep.infer_irreps(vib_modes.C2v_table)

result = branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)
ref = vib_modes.branching_change_of_basis(Td_irreps, C2v_irreps, _subs_elements)

# Compare shapes of all change-of-basis matrices
result_shapes = sorted([arr.shape for sublist in result for arr in sublist])
ref_shapes = sorted([arr.shape for sublist in ref for arr in sublist])
assert result_shapes == ref_shapes, f"Shape mismatch:\n  got {result_shapes}\n  expected {ref_shapes}"
print(f"branching_change_of_basis test passed! ({len(result_shapes)} matrices matched)")

---
## Section 6: Cartesian Tensors under Point Group Symmetry

Cartesian tensors can be decomposed into irreps of $O(3)$. When a physical system has a point group symmetry $G \subset O(3)$, only tensor components that transform as the **trivial representation** ($A_1$ or $A_{1g}$) of $G$ are non-zero.

**Workflow:**
1. Build $O(3)$ irrep representations (using `so3.o3_irrep`) restricted to the elements of a point group
2. Use `branching_change_of_basis` (or `vib_modes.branching_change_of_basis`) to decompose each $O(3)$ irrep into point group irreps
3. Count how many times the trivial irrep appears — those are the non-zero components

In [ ]:
# Setup: build point group data for Oh and D2h
# (SO(3) irreps up to L=4 are already in so3_irreps from the reference data cell)
from pymatgen.symmetry.groups import PointGroup

# Oh point group
Oh_group = PointGroup(vib_modes.point_group_dict['Oh'])
Oh_vec = np.stack([op.rotation_matrix for op in Oh_group.symmetry_ops], axis=0)
Oh_table = groups.make_multiplication_table(Oh_vec)

# D2h point group
D2h_group = PointGroup(vib_modes.point_group_dict['D2h'])
D2h_vec = np.stack([op.rotation_matrix for op in D2h_group.symmetry_ops], axis=0)
D2h_table = groups.make_multiplication_table(D2h_vec)

np.random.seed(42)
Oh_irreps = rep.infer_irreps(Oh_table)
D2h_irreps = rep.infer_irreps(D2h_table)

print(f"Oh: {len(Oh_vec)} elements, {len(Oh_irreps)} irreps (dims: {[ir.shape[-1] for ir in Oh_irreps]})")
print(f"D2h: {len(D2h_vec)} elements, {len(D2h_irreps)} irreps (dims: {[ir.shape[-1] for ir in D2h_irreps]})")

In [ ]:
# Helper: build O(3) irrep matrices for a point group's elements,
# then branch into point group irreps and count trivial multiplicities.
#
# Use this cell to explore which O(3) irreps become (partially) trivial
# under Oh and D2h.
#
# Workflow for each O(3) irrep (L, parity):
#   1. Compute rotation parameters for the point group elements
#   2. Compute inversion parameters (+1 for det=+1, -1 for det=-1)
#   3. Build O(3) irrep matrices using so3.o3_irrep
#   4. Branch into point group irreps using vib_modes.branching_change_of_basis
#   5. Check the multiplicity of the trivial irrep (first irrep, index 0)

# The course library provides so3.vec_3d_rep_to_rot_and_sign_params for
# extracting rotation and inversion parameters from 3x3 matrices.
rot_params_Oh, inv_params_Oh = so3.vec_3d_rep_to_rot_and_sign_params(Oh_vec)
rot_params_D2h, inv_params_D2h = so3.vec_3d_rep_to_rot_and_sign_params(D2h_vec)

# Verify: reconstruct the L=1 representation and check it matches
o3_1o_Oh = so3.o3_irrep(so3_irreps[1], rot_params_Oh, inv_params_Oh, parity=-1)
np.testing.assert_allclose(o3_1o_Oh, Oh_vec, atol=1e-7)
print("Rotation parameter extraction verified (Oh)!")

o3_1o_D2h = so3.o3_irrep(so3_irreps[1], rot_params_D2h, inv_params_D2h, parity=-1)
np.testing.assert_allclose(o3_1o_D2h, D2h_vec, atol=1e-7)
print("Rotation parameter extraction verified (D2h)!")

# YOUR EXPLORATION HERE:
# Build O(3) irreps for each (L, parity) pair, branch into Oh/D2h irreps,
# and check which contain the trivial representation.

### Irreps of $O(3)$ under $O_h$ and $D_{2h}$

Which of the following irreps of $O(3)$ (fully or partially) become trivial under $O_h$ / $D_{2h}$?

Options: `['0e', '0o', '1e', '1o', '2e', '2o', '3e', '3o', '4e', '4o']`

In [ ]:
# YOUR ANSWER HERE:
# Which O(3) irreps (partially) become trivial under Oh?
# o3_under_oh = [...]  # list of 'Lp' strings, e.g. ['0e', '4e']

# Which O(3) irreps (partially) become trivial under D2h?
# o3_under_d2h = [...]  # list of 'Lp' strings

# Why are certain irreps non-trivial under BOTH Oh and D2h?
# Think about what structural property these two groups share.

### Moment of Inertia Tensor (`ij=ji` for `i=1o`)

A moment of inertia tensor decomposes into $O(3)$ irreps `0e + 2e`.

How many components of each irrep are trivial (i.e., contain the trivial point group irrep) under $O_h$ and $D_{2h}$?

In [ ]:
# YOUR ANSWER HERE:
# Inertia tensor = 0e + 2e. How many trivial components under each group?
# Give answers as [count_for_0e, count_for_2e].

# inertia_oh = [...]   # under Oh
# inertia_d2h = [...]   # under D2h

### Elasticity Tensor (`ijkl=jikl=ijlk=klij` for `i=1o`)

An elasticity tensor decomposes into $O(3)$ irreps `2x0e + 2x2e + 1x4e`.

How many components of each irrep are trivial under $O_h$ and $D_{2h}$?

In [ ]:
# YOUR ANSWER HERE:
# Elasticity tensor = 2x0e + 2x2e + 1x4e. How many trivial components?
# Give answers as [count_for_0e, count_for_2e, count_for_4e].

# elasticity_oh = [...]   # under Oh
# elasticity_d2h = [...]   # under D2h

---
## Explore Further

In [ ]:
# Try decomposing other tensors or exploring other point groups!